# 1. Create HiSeq Object and initialize instruments. 

You can skip initializing the Cameras if you're only priming the pumps.

In [1]:
import pyseq
import threading

hs = pyseq.HiSeq() # Create HiSeq Object
# hs.initializeCams() # Initialize Cameras
hs.initializeInstruments() # Initialize all other instruments (stages, pumps, valve, etc.)

2026-05-18 12:12 - HiSeq::Initializing FPGA 
2026-05-18 12:12 - HiSeq::Initializing X & Y stages 
2026-05-18 12:13 - HiSeq::Initializing lasers 
2026-05-18 12:16 - HiSeq::Initializing pumps and valves 
2026-05-18 12:16 - HiSeq::Initializing optics and Z stages 
2026-05-18 12:18 - HiSeq::Syncing Y stage 
2026-05-18 12:18 - HiSeq::Initialized!


True

# 2.  Configure instrument

Configure valves, pumps and then move the stage out

Default `waste_port` = 1, `flowcells` = A and B

In [2]:
waste_port = 1

inlets = 2 # Use 2 port stage inlet, can 2 or 8. If 8 need a HiSeq 2000 or X flow cell
port_dict = {'waste': waste_port} # Map reagents to valve ports

# Comment out the configuration you DONT use
flowcells = {'A':None, 'B':None}
# flowcells = {'B':None}
# flowcells = {'A':None}

hs.move_inlet(inlets) # Move to 2 port stage inlet
for fc in flowcells.keys():
    # loop through flow cells
    hs.v24[fc].port_dict = port_dict # Update valves with port mapping
    hs.p[fc].update_limits(8) # Configure pumps to use 8 barrels per flow cell lane

# Move stage Out
hs.move_stage_out() 

2026-05-18 12:20 - HiSeq::Moving stage out 


# 3. Lock flow cells on to the stage

# 4.  Prime Pumps

**Place a waste container at the inlet of `waste_port` for each of the `flowcells`**

**Submerge outlet lines of pump in at least 100 mL of water**

Pumps `total_volume` (mL) of water from pump outlet through `flowcells` and out to `waste_port` inlet

Defaults: `total_volume` = 10 mL, `outflow` = 500 uL/min, `inflow` = 1000 uL/min

In [7]:
total_volume = 10 # mL 

vol = 2000 # uL, volume for each round of pumping,  max is 2000 uL 
outflow = 500 # uL/min, pump to flowcells,  stay between 100 and 500 uL/min
inflow = 1000 # uL/min, outlet of pumps to pump, stay between 400 and 1000 uL/min

# Move to waste_port
print(f'Moving to waste port: {waste_port}')
for fc in flowcells.keys():
    hs.v24[fc].move('waste')
   
# Prime pumps
total_volume = total_volume * 1000 # convert to uL
primed_vol = 0
while primed_vol < total_volume:
    # Loop until total volume has been primed through pumps
    _vol = total_volume - primed_vol - vol
    if _vol >= 0:
        _vol = vol
    else:
        _vol = abs(_vol)    
    primed_vol += _vol
    print(f"Priming {primed_vol} uL of total {total_volume} uL")
    
    # Pump from pump outlet through flowcells to waste_port inlet
    for fc in flowcells.keys():
        flowcells[fc] = threading.Thread(target = hs.p[fc].reverse_pump, args  = (_vol, inflow, outflow))
        flowcells[fc].start()
    for fc in flowcells.keys():
        flowcells[fc].join()

Moving to waste port: 9
Priming 2000 of 2000


# 5. (Optional) Prime all lines

**Place a waste container at the inlet of each port (1-24) for each of the flowcells**

In [3]:
volume = 1500 # uL, volume to prime each line with
outflow = 500 # uL/min, pump to flowcells,  stay between 100 and 500 uL/min
inflow = 1000 # uL/min, outlet of pumps to pump, stay between 400 and 1000 uL/min
ports = range(1,25) # ports to prime (1-24)

for p in ports:
    # Loop through all valve ports
    
    print(f"Priming port {p} with {volume} uL")
    
    # Move to port p
    for fc in flowcells.keys():
        flowcells[fc] = threading.Thread(target = hs.v24[fc].move, args  = (p,))
        flowcells[fc].start()
    for fc in flowcells.keys():
        flowcells[fc].join()
        
    # Pump from pump outlet through flowcells to port inlet
    for fc in flowcells.keys():
        flowcells[fc] = threading.Thread(target = hs.p[fc].reverse_pump, args  = (volume, inflow, outflow))
        flowcells[fc].start()
    for fc in flowcells.keys():
        flowcells[fc].join()

Priming port 9 with 1500 uL
Priming port 20 with 1500 uL
Priming port 21 with 1500 uL
Priming port 22 with 1500 uL
Priming port 23 with 1500 uL
Priming port 24 with 1500 uL


# 6. Reset
Moves stages & filters to safe position, turns off lasers

In [6]:
print("Reseting X-Stage")
hs.x.move(30000) # center x stage
print("Reseting Y-Stage")
hs.y.move(0) # center y stage,
hs.y.command("OFF") # turn off y stage
print("Reseting Z-Stage")
hs.z.move([0, 0, 0]) # lower tilt stage
print("Reseting Lasers and Filters")
for color in ["red", "green"]:
    hs.optics.move_ex(color, 'home') # home emission filters
    hs.lasers[color].turn_on(False) # Turn off lasers
print("Reset complete")

## If using HiSeq within 10 days after reset, leave the system on.
1. Leave flow cells locked on to the stage and submerge all line inlets in water to prevent them from drying out.

2. Close the stage door.

3. Idle the sequencer: ```pyseq -idle```

## If not using HiSeq in more than 10 day after reset, turn off the sytem.
1. Remove the flow cells from the stage and all reagents from the chiller

2. Close the stage door

3. Turn the power off to the HiSeq